# 05  -  Ad-hoc cost query (customer self-serve)

A customer-friendly notebook for exploring AI Function costs in your own workspace via [`system.billing.usage`](https://docs.databricks.com/en/admin/system-tables/billing.html). Set the widgets at the top, then run any of the queries below.

This notebook can be opened **interactively** (edit widgets, run any subset of cells) or run **as the last task of the bundle's pipeline job** (uses sensible widget defaults that produce account-wide output).

**What you can answer with it:**
1. *"How much did we spend on AI Functions last 30 days?"*
2. *"Which specific function (`ai_extract` vs `ai_parse_document` vs `ai_query`) is driving the cost?"*
3. *"Which jobs and which users are responsible?"*
4. *"What's the daily trend?"*
5. *"Per-row cost on our actual production workload?"*

**Pricing model reference (see [AI Functions pricing page](https://www.databricks.com/product/pricing/ai-functions)):**

| Family | Functions | Billing bucket |
|---|---|---|
| **Task-specific** (per-input flat DBU rate) | `ai_parse_document`, `ai_extract`, `ai_classify` | `billing_origin_product = 'AI_FUNCTIONS'` |
| **General-purpose** (compute-time / per-token) | `ai_query`, `ai_summarize`, `ai_translate`, `ai_gen`, `ai_mask`, `ai_fix_grammar`, `ai_analyze_sentiment`, etc. | `billing_origin_product = 'MODEL_SERVING'` + `offering_type = 'BATCH_INFERENCE'` |

**Public references:**
- [System billing tables schema](https://docs.databricks.com/en/admin/system-tables/billing.html)
- [Cost queries for AI Function workloads (canonical)](https://docs.databricks.com/aws/en/large-language-models/ai-functions#view-costs-for-ai-function-workloads)
- [Monitor model serving costs](https://docs.databricks.com/aws/en/admin/system-tables/model-serving-cost)
- [Job costs and performance with system tables](https://docs.databricks.com/aws/en/admin/system-tables/jobs-cost)


## Widgets  -  customer's filter inputs

The next cell defines all the filter widgets you can edit at the top of the notebook. Leave empty (or `0` for `workspace_id`) to disable that filter. You don't need to edit any code below this cell  -  every query reads these widgets.

**Parameters:**
- `start_date`, `end_date`  -  date range (defaults to last 30 days). Both are inclusive.
- `workspace_id`  -  `0` = all workspaces in the account; otherwise filter to one specific workspace.
- `team_tag`, `owner_email`, `cost_center_tag`, `job_id_filter`  -  optional. Leave blank to see account-wide.
- `function_filter`  -  `ALL`, `AI_FUNCTIONS_ONLY` (just task-specific), `MODEL_SERVING_ONLY` (just batch inference), or one of `AI_PARSE_DOCUMENT` / `AI_EXTRACT` / `AI_CLASSIFY`.


In [ ]:
import datetime as dt

today = dt.date.today()
default_start = (today - dt.timedelta(days=30)).isoformat()
default_end   = today.isoformat()

dbutils.widgets.text("start_date",       default_start,    "Start date (YYYY-MM-DD)")
dbutils.widgets.text("end_date",         default_end,      "End date (YYYY-MM-DD)")
dbutils.widgets.text("workspace_id",     "0",              "Workspace ID (0 = all)")
dbutils.widgets.text("team_tag",         "",               "Filter by custom_tags.team (blank = no filter)")
dbutils.widgets.text("owner_email",      "",               "Filter by custom_tags.Owner (blank = no filter)")
dbutils.widgets.text("cost_center_tag",  "",               "Filter by custom_tags.cost_center (blank = no filter)")
dbutils.widgets.text("job_id_filter",    "",               "Filter by usage_metadata.job_id (blank = all jobs)")
dbutils.widgets.dropdown(
    "function_filter", "ALL",
    ["ALL", "AI_FUNCTIONS_ONLY", "MODEL_SERVING_ONLY",
     "AI_PARSE_DOCUMENT", "AI_EXTRACT", "AI_CLASSIFY"],
    "Function filter"
)

# Empty widget values fall back to last-30-days defaults (job task passes "")
START_DATE       = dbutils.widgets.get("start_date").strip() or default_start
END_DATE         = dbutils.widgets.get("end_date").strip()   or default_end
WORKSPACE_ID     = dbutils.widgets.get("workspace_id").strip() or "0"
TEAM_TAG         = dbutils.widgets.get("team_tag").strip()
OWNER_TAG        = dbutils.widgets.get("owner_email").strip()
COST_CENTER_TAG  = dbutils.widgets.get("cost_center_tag").strip()
JOB_ID_FILTER    = dbutils.widgets.get("job_id_filter").strip()
FUNCTION_FILTER  = dbutils.widgets.get("function_filter") or "ALL"

# Build the WHERE-clause fragments once, reused by every query below.
def build_filters():
    parts = [f"u.usage_date BETWEEN DATE'{START_DATE}' AND DATE'{END_DATE}'"]
    if WORKSPACE_ID and WORKSPACE_ID != "0":
        parts.append(f"u.workspace_id = '{WORKSPACE_ID}'")
    if TEAM_TAG:
        parts.append(f"u.custom_tags.team = '{TEAM_TAG}'")
    if OWNER_TAG:
        parts.append(f"u.custom_tags.Owner = '{OWNER_TAG}'")
    if COST_CENTER_TAG:
        parts.append(f"u.custom_tags.cost_center = '{COST_CENTER_TAG}'")
    if JOB_ID_FILTER:
        parts.append(f"u.usage_metadata.job_id = '{JOB_ID_FILTER}'")

    if FUNCTION_FILTER == "ALL":
        parts.append("""(
            u.billing_origin_product = 'AI_FUNCTIONS'
            OR (u.billing_origin_product = 'MODEL_SERVING'
                AND u.product_features.model_serving.offering_type = 'BATCH_INFERENCE')
        )""")
    elif FUNCTION_FILTER == "AI_FUNCTIONS_ONLY":
        parts.append("u.billing_origin_product = 'AI_FUNCTIONS'")
    elif FUNCTION_FILTER == "MODEL_SERVING_ONLY":
        parts.append("u.billing_origin_product = 'MODEL_SERVING' AND u.product_features.model_serving.offering_type = 'BATCH_INFERENCE'")
    else:
        parts.append(f"u.billing_origin_product = 'AI_FUNCTIONS' AND u.product_features.ai_functions.ai_function = '{FUNCTION_FILTER}'")
    return " AND ".join(parts)

WHERE_CLAUSE = build_filters()

print("Active filters:")
print(f"  Date range:      {START_DATE} -> {END_DATE}")
print(f"  Workspace:       {WORKSPACE_ID if WORKSPACE_ID != '0' else 'ALL'}")
print(f"  team_tag:        {TEAM_TAG or '(any)'}")
print(f"  owner_email:     {OWNER_TAG or '(any)'}")
print(f"  cost_center_tag: {COST_CENTER_TAG or '(any)'}")
print(f"  job_id:          {JOB_ID_FILTER or '(any)'}")
print(f"  function_filter: {FUNCTION_FILTER}")


## Pre-flight check  -  does the filter set match any rows?

The next cell runs a single `COUNT(*)` against the filter set so you can confirm upfront whether queries below will be populated. If this prints `0`, every `display(...)` below will look empty too  -  change a widget and re-run.

This is the most common reason for "the notebook output looks blank": filters too narrow, or recent activity hasn't propagated past the [1-4 hour billing lag](https://docs.databricks.com/en/admin/system-tables/billing.html).


In [ ]:
preflight = spark.sql(f"""
    SELECT COUNT(*) AS billing_rows_matched
    FROM system.billing.usage u
    WHERE {WHERE_CLAUSE}
""").collect()[0]["billing_rows_matched"]

print(f"Billing rows matched by current filters: {preflight:,}")
if preflight == 0:
    print()
    print("Likely fixes:")
    print("  1. Clear the team_tag / owner_email / cost_center_tag widgets and re-run -- they may not have propagated to billing yet.")
    print("  2. Widen the date range (try 60 or 90 days).")
    print("  3. Switch function_filter to ALL.")
    print("  4. If the activity was within the last 1-4 hours, it may not be in system.billing.usage yet -- wait and re-run.")
else:
    print("OK -- queries below will return data.")


## Query 1  -  Headline total

The next cell answers: *"How much did we spend total on AI Functions in this window?"* It returns a single row with total USD and total DBUs.

The price-join uses [`system.billing.list_prices`](https://docs.databricks.com/en/admin/system-tables/billing.html#schema-2-billing-list_prices) to convert DBUs to USD using the price in effect at the time of usage (so a price change mid-window is handled correctly).


In [ ]:
display(spark.sql(f"""
    SELECT
        ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)), 2) AS total_usd,
        ROUND(SUM(u.usage_quantity), 2)                                    AS total_dbus,
        COUNT(*)                                                           AS billing_rows,
        COUNT(DISTINCT u.workspace_id)                                     AS workspaces_seen,
        MIN(u.usage_date)                                                  AS first_usage_date,
        MAX(u.usage_date)                                                  AS last_usage_date
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    WHERE {WHERE_CLAUSE}
"""))


## Query 2  -  Cost by function family

The next cell breaks the total down by which AI function family drove the cost. This is the most useful view to understand *what* you're being billed for. Each row is one of:

| Family label | Source columns |
|---|---|
| `AI Functions: ai_parse_document` | `billing_origin_product='AI_FUNCTIONS'` + `ai_function='AI_PARSE_DOCUMENT'` |
| `AI Functions: ai_extract` | `billing_origin_product='AI_FUNCTIONS'` + `ai_function='AI_EXTRACT'` |
| `AI Functions: ai_classify` | `billing_origin_product='AI_FUNCTIONS'` + `ai_function='AI_CLASSIFY'` |
| `Model Serving (batch): ai_query / ai_summarize / ai_translate / etc.` | `billing_origin_product='MODEL_SERVING'` + `offering_type='BATCH_INFERENCE'` |


In [ ]:
display(spark.sql(f"""
    SELECT
        CASE
            WHEN u.billing_origin_product = 'AI_FUNCTIONS' THEN
                CONCAT('AI Functions: ', LOWER(u.product_features.ai_functions.ai_function))
            WHEN u.billing_origin_product = 'MODEL_SERVING'
                 AND u.product_features.model_serving.offering_type = 'BATCH_INFERENCE' THEN
                'Model Serving (batch): ai_query / ai_gen / ai_summarize / etc.'
            ELSE u.billing_origin_product
        END                                                              AS function_family,
        ROUND(SUM(u.usage_quantity), 2)                                   AS dbus,
        ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)), 2) AS cost_usd,
        COUNT(*)                                                           AS billing_rows
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    WHERE {WHERE_CLAUSE}
    GROUP BY function_family
    ORDER BY cost_usd DESC
"""))


## Query 3  -  Daily trend

The next cell answers: *"Is the cost spiking on certain days?"* It returns one row per `usage_date` with daily total. Plot it as a line chart in the notebook UI to spot trends or anomalies.


In [ ]:
display(spark.sql(f"""
    SELECT
        u.usage_date,
        ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)), 2) AS cost_usd,
        ROUND(SUM(u.usage_quantity), 2)                                    AS dbus,
        COUNT(*)                                                           AS billing_rows
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    WHERE {WHERE_CLAUSE}
    GROUP BY u.usage_date
    ORDER BY u.usage_date
"""))


## Query 4  -  Top jobs (which scheduled jobs are spending the most?)

The next cell answers: *"Which job_ids are responsible for our AI Function bill?"* It joins to [`system.lakeflow.jobs`](https://docs.databricks.com/aws/en/admin/system-tables/jobs-cost) for human-readable job names.

Useful for: identifying the production fraud-detection job, finding rogue dev runs, or confirming that a single job is responsible for spend.


In [ ]:
display(spark.sql(f"""
    SELECT
        u.usage_metadata.job_id,
        COALESCE(j.name, '(no job name in system.lakeflow.jobs)')          AS job_name,
        u.workspace_id,
        ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)), 2)  AS cost_usd,
        ROUND(SUM(u.usage_quantity), 2)                                    AS dbus,
        COUNT(*)                                                           AS billing_rows
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    LEFT JOIN system.lakeflow.jobs j
      ON u.usage_metadata.job_id = j.job_id
      AND u.workspace_id          = j.workspace_id
    WHERE {WHERE_CLAUSE}
      AND u.usage_metadata.job_id IS NOT NULL
    GROUP BY u.usage_metadata.job_id, j.name, u.workspace_id
    ORDER BY cost_usd DESC
    LIMIT 25
"""))


## Query 5  -  Top tag values (who's spending  -  by team, owner, cost_center)

The next cell answers: *"Which team / owner / cost_center is spending the most?"*  -  useful when chargeback or budget allocation is by tag. Make sure your jobs are tagged with `custom_tags.team`, `custom_tags.cost_center`, and `custom_tags.Owner`. See the [tag setup docs](https://docs.databricks.com/en/admin/account-settings/usage-detail-tags.html).


In [ ]:
display(spark.sql(f"""
    SELECT
        u.custom_tags.team                                                 AS team,
        u.custom_tags.Owner                                                AS owner,
        u.custom_tags.cost_center                                          AS cost_center,
        ROUND(SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)), 2)  AS cost_usd,
        COUNT(*)                                                           AS billing_rows
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    WHERE {WHERE_CLAUSE}
    GROUP BY u.custom_tags.team, u.custom_tags.Owner, u.custom_tags.cost_center
    ORDER BY cost_usd DESC
    LIMIT 25
"""))


## Query 6  -  Latest 50 raw billing rows (for drill-down)

The next cell returns the most recent 50 billing rows matching your filters. Useful for sanity-checking that filters are doing what you expect, or for inspecting which exact `sku_name` / `endpoint_name` your workload is using.


In [ ]:
display(spark.sql(f"""
    SELECT
        u.usage_date,
        u.workspace_id,
        u.billing_origin_product,
        CASE
            WHEN u.billing_origin_product = 'AI_FUNCTIONS' THEN u.product_features.ai_functions.ai_function
            WHEN u.billing_origin_product = 'MODEL_SERVING' THEN u.product_features.model_serving.offering_type
            ELSE NULL
        END                                              AS detail,
        u.sku_name,
        u.usage_metadata.job_id,
        u.usage_metadata.endpoint_name,
        u.custom_tags.team,
        u.custom_tags.Owner,
        u.custom_tags.cost_center,
        ROUND(u.usage_quantity, 4)                       AS dbus,
        ROUND(u.usage_quantity * COALESCE(lp.pricing.default, 0), 6) AS cost_usd,
        u.usage_start_time,
        u.usage_end_time
    FROM system.billing.usage u
    LEFT JOIN system.billing.list_prices lp
      ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
      AND lp.currency_code = 'USD'
      AND u.usage_start_time >= lp.price_start_time
      AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
    WHERE {WHERE_CLAUSE}
    ORDER BY u.usage_start_time DESC
    LIMIT 50
"""))


## Query 7  -  Forecast extrapolation

The next cell takes the average cost-per-day in the window, then projects it out for 30 / 90 / 365 days. Useful for budget conversations: *"At current run-rate, what's our projected annual spend?"*


In [ ]:
display(spark.sql(f"""
    WITH daily AS (
        SELECT
            u.usage_date,
            SUM(u.usage_quantity * COALESCE(lp.pricing.default, 0)) AS daily_cost
        FROM system.billing.usage u
        LEFT JOIN system.billing.list_prices lp
          ON u.sku_name = lp.sku_name AND u.cloud = lp.cloud
          AND lp.currency_code = 'USD'
          AND u.usage_start_time >= lp.price_start_time
          AND (lp.price_end_time IS NULL OR u.usage_start_time < lp.price_end_time)
        WHERE {WHERE_CLAUSE}
        GROUP BY u.usage_date
    )
    SELECT
        ROUND(AVG(daily_cost), 2)         AS avg_daily_usd,
        ROUND(AVG(daily_cost) * 30, 2)    AS projected_30d_usd,
        ROUND(AVG(daily_cost) * 90, 2)    AS projected_90d_usd,
        ROUND(AVG(daily_cost) * 365, 2)   AS projected_365d_usd,
        COUNT(*)                          AS days_with_usage
    FROM daily
"""))


## Done  -  recommended customer next steps

After you've identified your high-cost AI function and team:

1. **Set a [budget alert](https://docs.databricks.com/en/admin/account-settings/budgets.html)** on `billing_origin_product IN ('AI_FUNCTIONS', 'MODEL_SERVING')` filtered by your `team_tag` or `cost_center_tag`. Get an email when forecasted month-end exceeds your threshold.

2. **Standardize tag hygiene**  -  every job that calls AI Functions should set `custom_tags.team`, `custom_tags.cost_center`, and `custom_tags.Owner` at job-definition time. See [usage detail tags](https://docs.databricks.com/en/admin/account-settings/usage-detail-tags.html). Without these, the team/owner queries return blanks.

3. **Pick the right function for the workload**  -  `ai_extract` (per-input flat rate) is more predictable for production extraction; `ai_query` (compute-time) is more flexible but variance is harder to forecast. Run the bundle's `ai_functions_cost_demo` job in your environment to compare on your real data shape.

4. **Watch for the [50% AI Functions promo](https://www.databricks.com/product/pricing/ai-functions) ending 2026-06-30**  -  list price kicks in on July 1.
